# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Approach.** The ranked queue comes from the ML-08 Logistic Regression, scored only on the client-held-out test split (8 clients, 7,115 rows never seen during training) — the same split validated in ML-09. Ranking uses `pred_proba` (probability of `is_declining_label`), and priority is capped at the top 200 rows, because K=200 is the deepest point actually validated (precision@200 = 0.705 against a 0.517 base rate). Ranking further down the list moves past what's been checked, so it isn't included as an actioned row — it falls into `stable_low_priority` by default instead.

Reason codes and archetypes come from the error-analysis pattern found in ML-08 §4, not from a separate clustering step — this repo's model is a classifier, not a segmentation model:

- **`consistent_fade`** — inside the validated top 200, low day-coverage (`days_with_impressions`/`days_with_sessions` both low). Straightforward decline signal.
- **`visibility_without_clicks`** — inside the top 200, but with a wide gap between `days_with_impressions` and `days_with_sessions` (top quartile of the gap). ML-08's error analysis found this exact shape produces false positives — impressions without matching sessions look like decline to the model but are often a CTR/snippet problem, not a content-decline problem.
- **`large_page_blindspot`** — outside the top 200 (model says low risk) but in the top 10% of `users_90d` with a strong average position (0 < position ≤ 10). ML-08's false-negative example was exactly this shape: 83,603 impressions, position 3.4, 802 users — model scored it low risk, label said declining. This archetype exists because the model reads current scale, not direction of change; it needs a human-set override lane the score alone won't produce.
- **`stable_low_priority`** — everything else. No action this cycle.

In [1]:
# --- Same repo, same pipeline as ML-08/ML-09, rebuilt here for the playbook ---
!git clone https://github.com/HasnainRaza16/flyrank-ml-Hasnain.git
%cd flyrank-ml-Hasnain


Cloning into 'flyrank-ml-Hasnain'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 182 (delta 79), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 1.91 MiB | 7.75 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/flyrank-ml-Hasnain


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Same safe feature set as ML-08/ML-09 — current-window, non-label, non-ID
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

df[numeric_features] = df[numeric_features].fillna(0)
df[categorical_features] = df[categorical_features].fillna("unknown")

X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
y = df["is_declining_label"]
groups = df["client_id"]

# Grouped split by client — matches ML-08/ML-09, never mixes a client across train/test
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
logreg.fit(X_train_s, y_train)
proba_test = logreg.predict_proba(X_test_s)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

auc = roc_auc_score(y_test, proba_test)
print("test rows:", len(y_test), "| test clients:", df.iloc[test_idx]["client_id"].nunique())
print("ROC AUC (client-held-out):", round(auc, 3), "| test base rate:", round(y_test.mean(), 3))
print("precision@200 (validated in ML-08/ML-09):", round(precision_at_k(proba_test, y_test.values, 200), 3))


test rows: 7115 | test clients: 8
ROC AUC (client-held-out): 0.589 | test base rate: 0.517
precision@200 (validated in ML-08/ML-09): 0.705


In [3]:
# --- Build the ranked queue, capped at the validated K=200, with archetypes + reason codes ---
queue = df.iloc[test_idx][[
    "content_id", "content_type", "main_intent", "impressions_90d", "avg_position",
    "days_with_impressions", "days_with_sessions", "users_90d", "content_age_days",
    "days_since_last_update"
]].copy().reset_index(drop=True)
queue["pred_proba"] = proba_test
queue["consistency_gap"] = queue["days_with_impressions"] - queue["days_with_sessions"]

queue = queue.sort_values("pred_proba", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

TOP_K = 200  # the deepest K actually validated (precision@200 = 0.705)
GAP_P75 = queue["consistency_gap"].quantile(0.75)
SCALE_P90 = queue["users_90d"].quantile(0.90)

def classify(row):
    if row["rank"] <= TOP_K:
        if row["consistency_gap"] >= GAP_P75:
            return "visibility_without_clicks", "top200_by_score_but_wide_impression_session_gap"
        return "consistent_fade", "top200_by_score_low_day_coverage"
    if row["users_90d"] >= SCALE_P90 and 0 < row["avg_position"] <= 10:
        return "large_page_blindspot", "outside_top200_but_matches_known_false_negative_profile"
    return "stable_low_priority", "outside_top200_no_flag_pattern"

queue[["archetype", "reason_code"]] = queue.apply(lambda r: pd.Series(classify(r)), axis=1)

action_map = {
    "consistent_fade": "Refresh content: update facts/data, re-check target keyword, add current context.",
    "visibility_without_clicks": "Check title tag, meta description, and SERP snippet before refreshing body content — the gap suggests a click-through problem, not necessarily a content-decline problem.",
    "large_page_blindspot": "Spot-check manually on a rolling basis even though the score is low — this profile matched pages the model missed in ML-08's error analysis.",
    "stable_low_priority": "No action this cycle. Re-score next window.",
}
queue["suggested_action"] = queue["archetype"].map(action_map)

print(queue["archetype"].value_counts())
print()
print("mean pred_proba by archetype:")
print(queue.groupby("archetype")["pred_proba"].mean().round(3))


archetype
stable_low_priority          6392
large_page_blindspot          523
visibility_without_clicks     119
consistent_fade                81
Name: count, dtype: int64

mean pred_proba by archetype:
archetype
consistent_fade              0.865
large_page_blindspot         0.522
stable_low_priority          0.575
visibility_without_clicks    0.862
Name: pred_proba, dtype: float64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** This queue is decision-support for a content reviewer deciding which existing pages to open first this cycle. It ranks against an observed, already-computed label (`trend_direction`) — it is **not** a forecast of future decline, and it should not auto-trigger content changes without the human review step in §3.

**Validated scope.** The precision numbers above (0.85 / 0.80 / 0.74 / 0.705 at K = 20 / 50 / 100 / 200) hold on clients the model never saw during training — 8 held-out clients, 7,115 rows, from the 30k-row anonymized teaching slice. Two things narrow this further:
1. The random-split number for this same model is 0.693 ROC AUC, 0.10 higher than the honest, client-held-out number (0.589). ML-09 showed this gap is real, so any accuracy figure pulled from an ungrouped split — including the paper's own growth-model finding — should not be taken at face value.
2. This was validated on 32 clients total, 8 held out. It has not been checked against a client whose content mix, industry, or scale looks very different from these 32 — treat a genuinely new client as out of validated scope until re-checked.

**Limits, in the claim ladder from `writing-honest-claims`:**
- *Observed:* the label is a computed past outcome (`trend_direction == "down"`), not a prediction of what happens next.
- *Measured:* ROC AUC 0.589 against a 0.517 base rate, client-held-out — a real but modest lift over guessing.
- *Directional:* the model ranks pages better than random at surfacing already-declined pages; it does not explain why beyond the top features, and it is not "accurate" in the everyday sense of that word.
- *Decision-support only:* every flagged row still needs the human check in §3 before any action is taken.

The sentence this playbook does **not** use: "the model predicts which pages will decline." The sentence it uses instead (from ML-09's own rewrite): the model's ranking is directionally better than random at surfacing pages with an *already-observed* decline, on clients it hasn't seen — it doesn't forecast the future, and its lift over guessing is modest, not dramatic.

In [4]:
# Quick-reference metrics block for anyone reading this notebook without re-running it
validated_metrics = {
    "roc_auc_client_held_out": round(float(auc), 3),
    "roc_auc_random_split_do_not_use": 0.693,
    "test_base_rate": round(float(y_test.mean()), 3),
    "precision_at_20": round(precision_at_k(proba_test, y_test.values, 20), 3),
    "precision_at_50": round(precision_at_k(proba_test, y_test.values, 50), 3),
    "precision_at_100": round(precision_at_k(proba_test, y_test.values, 100), 3),
    "precision_at_200": round(precision_at_k(proba_test, y_test.values, 200), 3),
    "validated_on_n_clients": int(df.iloc[test_idx]["client_id"].nunique()),
    "total_clients_in_dataset": int(df["client_id"].nunique()),
}
for k, v in validated_metrics.items():
    print(f"{k}: {v}")


roc_auc_client_held_out: 0.589
roc_auc_random_split_do_not_use: 0.693
test_base_rate: 0.517
precision_at_20: 0.85
precision_at_50: 0.8
precision_at_100: 0.74
precision_at_200: 0.705
validated_on_n_clients: 8
total_clients_in_dataset: 32


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a person must check before acting on a flagged row:**
1. Open the page and check the SERP snippet, title tag, and meta description before assuming a content problem — `visibility_without_clicks` rows are the clearest case where the fix might be a title, not a rewrite.
2. Confirm `avg_position` isn't `0` (that means *no data* in this dataset, not rank zero — per the data dictionary) before trusting a `large_page_blindspot` flag that depends on it.
3. Check whether the page's client resembles the 32 in this dataset. If it's a client type the model hasn't seen, treat the score as unvalidated and defer to manual judgment (§2).
4. Spot-check a sample of `large_page_blindspot` rows even though the model didn't flag them — this is the one profile with a demonstrated blind spot, not a hypothetical one.

**No-go list — what should never be automated from this model alone:**
- No automatic content deletion, de-indexing, or unpublishing based on `pred_proba` alone.
- No automatic publishing of rewritten content without a human reading the draft first.
- No client-facing report stating this model "predicts" decline — external summaries should use the §2 claim-ladder language, not a shortcut version of it.
- No scoring of a client type outside the 32 in this dataset without flagging the result as unvalidated.

In [5]:
no_go_list = [
    "auto_delete_or_deindex_content",
    "auto_publish_rewrites_without_human_read",
    "external_report_claims_prediction_of_future_decline",
    "score_unvalidated_client_type_without_flagging_it",
]
human_review_checklist = [
    "check_serp_snippet_title_meta_before_assuming_content_problem",
    "confirm_avg_position_is_not_the_no_data_placeholder",
    "confirm_client_resembles_the_32_validated_clients",
    "spot_check_large_page_blindspot_rows_regardless_of_score",
]
print("NO-GO:")
for item in no_go_list:
    print(" -", item)
print()
print("HUMAN REVIEW CHECKLIST:")
for item in human_review_checklist:
    print(" -", item)


NO-GO:
 - auto_delete_or_deindex_content
 - auto_publish_rewrites_without_human_read
 - external_report_claims_prediction_of_future_decline
 - score_unvalidated_client_type_without_flagging_it

HUMAN REVIEW CHECKLIST:
 - check_serp_snippet_title_meta_before_assuming_content_problem
 - confirm_avg_position_is_not_the_no_data_placeholder
 - confirm_client_resembles_the_32_validated_clients
 - spot_check_large_page_blindspot_rows_regardless_of_score


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**What would tell us the recommendations went stale:**
- **Base-rate drift.** This model was validated against a ~0.52 base rate for `is_declining_label`. If a fresh batch's base rate moves far from that, precision@K stops meaning what it meant here — recheck before trusting the ranking.
- **Precision@K decay.** Recompute precision@50/100/200 each cycle on freshly labeled rows (once `trend_direction` is observable for them). A sustained drop below the validated numbers (0.80 / 0.74 / 0.705) is the trigger to retrain, not just re-score.
- **Feature drift.** The top signals (`days_with_impressions`, `days_with_sessions`, `users_90d`) are consistency/scale measures. If a client's CMS or analytics setup changes how these get reported, the model's meaning shifts underneath it — re-run the leakage/importance check from ML-08/ML-09 before trusting new rankings.
- **New-client coverage.** Any client outside the 32 in this dataset is out-of-scope by default (§2) — track this count explicitly instead of silently scoring new clients the same way as validated ones.

In [6]:
# Retrain-trigger thresholds, checked against this run's own numbers as a worked example.
# In production, re-run this cell against a FRESH labeled batch, not the same test set used to validate.
RETRAIN_TRIGGERS = {
    "base_rate_drift_max_abs_change": 0.05,   # trigger if |new_base_rate - 0.517| exceeds this
    "min_acceptable_precision_at_50": 0.70,   # below validated 0.80 minus a margin
    "min_acceptable_precision_at_100": 0.65,  # below validated 0.74 minus a margin
    "min_acceptable_precision_at_200": 0.60,  # below validated 0.705 minus a margin
}

current_run = {
    "base_rate": round(float(y_test.mean()), 3),
    "precision_at_50": round(precision_at_k(proba_test, y_test.values, 50), 3),
    "precision_at_100": round(precision_at_k(proba_test, y_test.values, 100), 3),
    "precision_at_200": round(precision_at_k(proba_test, y_test.values, 200), 3),
}

BASELINE_BASE_RATE = 0.517
print("base_rate_drift:", round(abs(current_run["base_rate"] - BASELINE_BASE_RATE), 3),
      "-> RETRAIN" if abs(current_run["base_rate"] - BASELINE_BASE_RATE) > RETRAIN_TRIGGERS["base_rate_drift_max_abs_change"] else "-> ok")
print("precision_at_50:", current_run["precision_at_50"],
      "-> RETRAIN" if current_run["precision_at_50"] < RETRAIN_TRIGGERS["min_acceptable_precision_at_50"] else "-> ok")
print("precision_at_100:", current_run["precision_at_100"],
      "-> RETRAIN" if current_run["precision_at_100"] < RETRAIN_TRIGGERS["min_acceptable_precision_at_100"] else "-> ok")
print("precision_at_200:", current_run["precision_at_200"],
      "-> RETRAIN" if current_run["precision_at_200"] < RETRAIN_TRIGGERS["min_acceptable_precision_at_200"] else "-> ok")


base_rate_drift: 0.0 -> ok
precision_at_50: 0.8 -> ok
precision_at_100: 0.74 -> ok
precision_at_200: 0.705 -> ok


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**What gets exported and why.** The paper's recommendations section builds on two things: the ranked queue itself, and the numbers that back its claims. The queue CSV goes to `work/outputs/` (regenerated by this notebook, kept out of git by the CI leak-guard by design). The archetype figure goes to `work/figures/` (committed — it's a chart, not raw client data). The metrics JSON also goes to `work/outputs/`, but per the assignment note it should be committed as a receipt — if the leak-guard's `.gitignore` pattern blocks it too, use `git add -f work/outputs/metrics.json` when committing.

In [7]:
import os
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1) Ranked queue (client_id excluded — pseudonym, not needed downstream)
export_cols = [
    "content_id", "content_type", "main_intent", "rank", "pred_proba",
    "archetype", "reason_code", "suggested_action",
    "impressions_90d", "avg_position", "days_with_impressions", "days_with_sessions", "users_90d",
]
queue[export_cols].to_csv("work/outputs/w07_content_action_queue.csv", index=False)
print("wrote work/outputs/w07_content_action_queue.csv:", len(queue), "rows")

# 2) Figure — archetype distribution, reused in the paper
counts = queue["archetype"].value_counts().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(counts.index, counts.values, color="#4C72B0")
ax.set_xlabel(f"Number of pages (client-held-out test set, n={len(queue)})")
ax.set_title("Content action queue — pages by archetype")
for i, v in enumerate(counts.values):
    ax.text(v + 30, i, str(v), va="center")
plt.tight_layout()
plt.savefig("work/figures/w07_archetype_counts.png", dpi=150)
plt.close()
print("wrote work/figures/w07_archetype_counts.png")

# 3) Metrics JSON — the receipts the paper's numbers trace back to
metrics = {
    "model": "LogisticRegression (won precision@K comparison vs RandomForest, ML-08)",
    "split": "GroupShuffleSplit by client_id, test_size=0.25, random_state=42",
    "n_test_rows": int(len(y_test)),
    "n_test_clients": int(df.iloc[test_idx]["client_id"].nunique()),
    "total_clients_in_dataset": int(df["client_id"].nunique()),
    "roc_auc_client_held_out": round(float(auc), 3),
    "roc_auc_random_split_inflated_do_not_use": 0.693,
    "test_base_rate": round(float(y_test.mean()), 3),
    "precision_at_k": {str(k): round(precision_at_k(proba_test, y_test.values, k), 3) for k in [20, 50, 100, 200]},
    "top_features_permutation_importance": [
        "days_with_impressions", "days_with_sessions", "users_90d", "content_age_days", "avg_position"
    ],
    "known_failure_modes": [
        "false_positive: high days_with_impressions, low days_with_sessions/users -- impression-without-click pages read as declining",
        "false_negative: high volume/position/users but actually declining -- model reads current scale, not direction of change",
    ],
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/w07_metrics.json")


wrote work/outputs/w07_content_action_queue.csv: 7115 rows
wrote work/figures/w07_archetype_counts.png
wrote work/outputs/w07_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.